In [1]:
import pandas as pd
import numpy as np
import optuna
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold

c:\Users\ardel\.conda\envs\ml_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

data_path = 'dataset_model/dataset_trainable_rev1.csv'

In [8]:
# ==============================================================================
# 1. LOAD DATASET & FILTER KONTAMINASI PENGANGKUTAN SAMPAH (SELF-REVISION)
# ==============================================================================
df = pd.read_csv(data_path)
print(f"Total baris awal: {len(df):,} baris")

# Hitung selisih target dan lag ke-3 untuk mendeteksi event pickup (> 20% drop)
target_diff = df['target_next_hour'] - df['fill_level_percent']
velocity_3h = df['lag_2h'] - df['lag_3h']

# Hanya ambil fase akumulasi murni (bebas dari drop pengangkutan sampah)
clean_mask = (
    (target_diff > -20.0) & 
    (df['velocity_1h'] > -20.0) & 
    (df['velocity_2h'] > -20.0) & 
    (velocity_3h > -20.0)
)

df_clean = df[clean_mask].reset_index(drop=True)
print(f"Total baris setelah pembersihan event pickup: {len(df_clean):,} baris (Dibuang: {len(df) - len(df_clean)} baris)")

Total baris awal: 4,577 baris
Total baris setelah pembersihan event pickup: 4,267 baris (Dibuang: 310 baris)


In [14]:
df.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1736024400,50.850685,52.753166,50.935368,49.665940,-1.902481,1.817799,0.707107,7.071068e-01,-0.781831,0.62349,53.736298
1,0,1736028000,53.736298,50.850685,52.753166,50.935368,2.885613,-1.902481,0.866025,5.000000e-01,-0.781831,0.62349,54.325520
2,0,1736031600,54.325520,53.736298,50.850685,52.753166,0.589222,2.885613,0.965926,2.588190e-01,-0.781831,0.62349,53.611057
3,0,1736035200,53.611057,54.325520,53.736298,50.850685,-0.714462,0.589222,1.000000,6.123234e-17,-0.781831,0.62349,55.610485
4,0,1736038800,55.610485,53.611057,54.325520,53.736298,1.999428,-0.714462,0.965926,-2.588190e-01,-0.781831,0.62349,56.233257


In [15]:
df_clean.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1736024400,50.850685,52.753166,50.935368,49.665940,-1.902481,1.817799,0.707107,7.071068e-01,-0.781831,0.62349,53.736298
1,0,1736028000,53.736298,50.850685,52.753166,50.935368,2.885613,-1.902481,0.866025,5.000000e-01,-0.781831,0.62349,54.325520
2,0,1736031600,54.325520,53.736298,50.850685,52.753166,0.589222,2.885613,0.965926,2.588190e-01,-0.781831,0.62349,53.611057
3,0,1736035200,53.611057,54.325520,53.736298,50.850685,-0.714462,0.589222,1.000000,6.123234e-17,-0.781831,0.62349,55.610485
4,0,1736038800,55.610485,53.611057,54.325520,53.736298,1.999428,-0.714462,0.965926,-2.588190e-01,-0.781831,0.62349,56.233257


In [9]:
# ==============================================================================
# 2. DEFINISI FITUR & TRAIN-TEST SPLIT
# ==============================================================================
features = [
    'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h',
    'velocity_1h', 'velocity_2h', 
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]
target = 'target_next_hour'

train_mask = df_clean['bin_index'] < 40
test_mask = df_clean['bin_index'] >= 40

X_train = df_clean.loc[train_mask, features]
y_train = df_clean.loc[train_mask, target]
groups_train = df_clean.loc[train_mask, 'bin_index']

X_test = df_clean.loc[test_mask, features]
y_test = df_clean.loc[test_mask, target]

# Evaluasi Baseline LightGBM (Default) pada data bersih
base_lgbm = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
base_lgbm.fit(X_train, y_train)
pred_base = base_lgbm.predict(X_test)
print(f"\n[Baseline LightGBM - Data Bersih] MAE: {mean_absolute_error(y_test, pred_base):.4f}%")


[Baseline LightGBM - Data Bersih] MAE: 1.7854%


In [12]:
# ==============================================================================
# 3. HYPERPARAMETER TUNING DENGAN OPTUNA (.SQLITE PERSISTENCE) + GROUP K-FOLD
# ==============================================================================
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        # num_leaves dibatasi maksimal 63 agar tidak menghafal noise pada 3.397 baris
        'num_leaves': trial.suggest_int('num_leaves', 7, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        # min_child_samples menjaga agar cabang pohon minimal mewakili 15 sampel nyata
        'min_child_samples': trial.suggest_int('min_child_samples', 15, 120),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 50.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 50.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    gkf = GroupKFold(n_splits=5)
    cv_maes = []
    
    for tr_idx, val_idx in gkf.split(X_train, y_train, groups=groups_train):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        cv_maes.append(mean_absolute_error(y_va, preds))
        
    return np.mean(cv_maes)

# Konfigurasi SQLite Storage agar studi bisa di-pause dan dilanjutkan kapan saja
db_filename = "sqlite:///optuna_smartbin_study.sqlite3"
study_name = "lgbm_smartbin_v1"

study = optuna.create_study(
    study_name=study_name,
    storage=db_filename,
    direction='minimize',
    load_if_exists=True  # Kunci utama agar otomatis resume jika file sudah ada
)

trials_existing = len(study.trials)
trials_to_add = 200  # Jumlah trial yang ingin dijalankan pada batch ini

print(f"Status Database '{study_name}': Ditemukan {trials_existing} trials sebelumnya.")
print(f"Menjalankan tambahan {trials_to_add} trials baru...")

study.optimize(objective, n_trials=trials_to_add, show_progress_bar=True)

print("\n=== STATUS OPTUNA DATABASE ===")
print(f"Total akumulasi trials tersimpan : {len(study.trials)} trials")
print(f"Best Cross-Validation MAE        : {study.best_value:.4f}%")
print("Parameter Terbaik Saat Ini       :", study.best_params)

Status Database 'lgbm_smartbin_v1': Ditemukan 0 trials sebelumnya.
Menjalankan tambahan 200 trials baru...


Best trial: 154. Best value: 1.83405: 100%|██████████| 200/200 [03:10<00:00,  1.05it/s]


=== STATUS OPTUNA DATABASE ===
Total akumulasi trials tersimpan : 200 trials
Best Cross-Validation MAE        : 1.8340%
Parameter Terbaik Saat Ini       : {'n_estimators': 457, 'learning_rate': 0.01736213165357414, 'num_leaves': 11, 'max_depth': 10, 'min_child_samples': 15, 'subsample': 0.5775574293685631, 'subsample_freq': 1, 'colsample_bytree': 0.897131757953279, 'reg_alpha': 0.00418197415628572, 'reg_lambda': 2.004523900477203e-08}


In [16]:
# ==============================================================================
# 4. EVALUASI & SIMPAN MODEL DARI DATABASE TERBAIK
# ==============================================================================
best_params = study.best_params.copy()
best_params.update({'random_state': 42, 'n_jobs': -1, 'verbose': -1})

final_model = LGBMRegressor(**best_params)
final_model.fit(X_train, y_train)

final_preds = final_model.predict(X_test)
final_mae = mean_absolute_error(y_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))
high_fill_mask = X_test['fill_level_percent'] >= 75.0
high_fill_mae = mean_absolute_error(y_test[high_fill_mask], final_preds[high_fill_mask])

print("\n=== HASIL EVALUASI MODEL FINAL (DARI SQLITE STUDY) ===")
print(f"Overall MAE (Semua Kondisi)      : {final_mae:.4f}%")
print(f"Overall RMSE                     : {final_rmse:.4f}%")
print(f"High-Fill MAE (Kapasitas >= 75%) : {high_fill_mae:.4f}%")

joblib.dump(final_model, 'lightgbm_smartbin_final.pkl')
print("\nModel berhasil disimpan sebagai 'lightgbm_smartbin_final.pkl'")


=== HASIL EVALUASI MODEL FINAL (DARI SQLITE STUDY) ===
Overall MAE (Semua Kondisi)      : 1.7493%
Overall RMSE                     : 2.2078%
High-Fill MAE (Kapasitas >= 75%) : 3.1627%

Model berhasil disimpan sebagai 'lightgbm_smartbin_final.pkl'
